# Infosys Quarterly Report Analysis using RAG

We will use the following libraries. Please install it in your virtual environment using `pip install <package_name>`

- **getpass4**
- **pypdf** 
- **faiss-cpu** 
- **llama-index** 
- **llama-index-readers-file** 
- **llama-index-vector-stores-faiss**  
- **llama-index-llms-groq** 
- **llama-index-embeddings-huggingface**
- **docling**

In [1]:
import os
from getpass import getpass
os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

In [3]:
from llama_index.core import SimpleDirectoryReader, ServiceContext, VectorStoreIndex, StorageContext
from llama_index.core.response.pprint_utils import pprint_response
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.core.query_engine import SubQuestionQueryEngine
from llama_index.core.node_parser import SimpleNodeParser
from llama_index.core.node_parser import (SentenceWindowNodeParser,)
from llama_index.core.text_splitter import SentenceSplitter
from llama_index.core import Document
import faiss
from llama_index.vector_stores.faiss import FaissVectorStore
from llama_index.llms.groq import Groq
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

## Configure LLM service

In [4]:
llm = Groq(model="llama-3.3-70b-versatile", temperature=0, max_tokens=512)

We will use [Qwen/Qwen3-Embedding-0.6B](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B) for embeddings.

In [5]:
embed_model = HuggingFaceEmbedding(model_name="Qwen/Qwen3-Embedding-0.6B")

In [6]:
from llama_index.core import Settings

Settings.llm = llm
Settings.embed_model = embed_model
Settings.node_parser = SentenceSplitter(chunk_size=512, chunk_overlap=20)
Settings.num_output = 512
Settings.context_window = 2048

## Load data
Downloaded from

https://www.infosys.com/investors/reports-filings/quarterly-results.html

In [8]:
from docling.document_converter import DocumentConverter

In [9]:
converter = DocumentConverter()
result = converter.convert("./ifrs-inr-press-release.pdf")
result.document.export_to_markdown()

# Define the output directory
output_dir = "output_documents"
os.makedirs(output_dir, exist_ok=True)  # Ensure the directory exists

# Define output file path
output_path = os.path.join(output_dir, "converted_document.md")

# Save as Markdown
with open(output_path, "w", encoding="utf-8") as f:
    f.write(result.document.export_to_markdown())

print(f"Document saved at: {output_path}")


2025-10-26 11:26:45,929 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]


2025-10-26 11:26:45,973 - INFO - Going to convert document batch...
2025-10-26 11:26:45,974 - INFO - Initializing pipeline for StandardPdfPipeline with options hash 4f2edc0f7d9bb60b38ebfecf9a2609f5
2025-10-26 11:26:46,012 - INFO - Loading plugin 'docling_defaults'
2025-10-26 11:26:46,015 - INFO - Registered picture descriptions: ['vlm', 'api']
2025-10-26 11:26:46,057 - INFO - Loading plugin 'docling_defaults'
2025-10-26 11:26:46,067 - INFO - Registered ocr engines: ['auto', 'easyocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']
2025-10-26 11:26:46,070 - INFO - rapidocr cannot be used because onnxruntime is not installed.
2025-10-26 11:26:46,071 - INFO - easyocr cannot be used because it is not installed.
2025-10-26 11:26:46,352 - INFO - Accelerator device: 'cuda:0'
[INFO] 2025-10-26 11:26:46,377 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2025-10-26 11:26:46,401 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\Sourav Karmakar\Desktop\Work\LogicMojo\logi

Document saved at: output_documents\converted_document.md


In [10]:
q1_2025 = SimpleDirectoryReader(
    input_files=["./output_documents/converted_document.md"]
).load_data()

# Build indices

In [11]:
# dimensions of embedding 
d = 1024
faiss_index = faiss.IndexFlatL2(d)

In [12]:
vector_store = FaissVectorStore(faiss_index=faiss_index)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
q1_2025_index = VectorStoreIndex.from_documents(q1_2025, storage_context=storage_context)

## Build query engines

In [13]:
q1_2025_engine = q1_2025_index.as_query_engine(similarity_top_k=3)

## Run queries

In [14]:
response = q1_2025_engine.query(
    "What is the revenue growth for the quarter?"
)

2025-10-26 11:32:28,383 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


In [15]:
pprint_response(response)

Final Response: The revenue growth for the quarter is 3.8% year-over-
year and 2.6% sequentially in constant currency. Reported revenues
grew by 7.5% year-over-year.


In [16]:
response = q1_2025_engine.query("Which are some of the key customer wins?")

2025-10-26 11:33:42,756 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


In [17]:
pprint_response(response)

Final Response: Some of the key customer wins include Zand Bank,
ALEXBANK Egypt, Export Development Bank of Egypt, and Agricultural
Bank of Egypt, as well as a $3.8 billion large deal win with 55% net
new clients.


In [18]:
response = q1_2025_engine.query("What are some of the key achievements in the quarter?")

2025-10-26 11:35:26,026 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


In [19]:
pprint_response(response)

Final Response: Some of the key achievements in the quarter include
revenues growing by 3.8% year-over-year and 2.6% sequentially in
constant currency, operating margin at 20.8%, and large deal wins with
a total contract value of $3.8 billion, of which 55% were net new.
Additionally, the company reported a strong free cash flow generation
and an increase in basic EPS by 8.6% year-over-year.


In [20]:
response = q1_2025_engine.query("What is the total assets as of June 2025?")

2025-10-26 11:36:10,302 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2025-10-26 11:36:10,536 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2025-10-26 11:36:10,735 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2025-10-26 11:36:11,213 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


In [21]:
pprint_response(response)

Final Response: The total assets as of June 2025 is 149,619 (in ₹
crore).
